[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoAvancadoIME/blob/main/03_deteccao_de_outliers.ipynb)

# Aula 3 — Detecção de Outliers e Ajustamento Robusto

**Maj Diego - 1º Semestre / 2027**

**Objetivos:**

1. Aplicar data snooping e o teste de Baarda.
2. Distinguir detecção, identificação e mitigação de outliers.
3. Implementar um ajuste robusto e comparar as normas L1 e L2.
4. Preparar a 2ª VE prática.

## 1. Data snooping e teste de Baarda

Após o teste global, analisam-se resíduos padronizados $w_i=v_i/\sigma_{v_i}$. Valores incompatíveis com o nível de significância indicam observações a investigar; a remoção nunca deve ser automática ou apenas numérica.

In [1]:
import numpy as np

x = np.arange(6, dtype=float)
y = np.array([1.0, 2.1, 3.0, 8.2, 5.1, 6.0])
A = np.column_stack([np.ones_like(x), x])
X = np.linalg.lstsq(A, y, rcond=None)[0]
v = A @ X - y
H = A @ np.linalg.inv(A.T @ A) @ A.T
qv = np.diag(np.eye(len(y)) - H)
s0 = np.sqrt((v @ v) / (len(y) - A.shape[1]))
w = v / (s0 * np.sqrt(qv))
print(np.column_stack([x, y, v, w]))

[[ 0.          1.          0.43333333  0.33350566]
 [ 1.          2.1         0.45333333  0.28679258]
 [ 2.          3.          0.67333333  0.39513637]
 [ 3.          8.2        -3.40666667 -1.99915528]
 [ 4.          5.1         0.81333333  0.51453962]
 [ 5.          6.          1.03333333  0.79528273]]


## 2. Ajustamento robusto

Estimadores robustos limitam a influência de resíduos grandes. No IRLS, uma sequência de MMQs reponderados aproxima perdas como Huber. A robustez reduz o efeito do outlier, mas não substitui sua investigação.

## 3. Normas L1 e L2

A norma **L2** minimiza $\sum v_i^2$, é eficiente sob erros gaussianos e sensível a extremos. A norma **L1** minimiza $\sum|v_i|$, tolera melhor outliers e produz um problema de otimização não diferenciável na origem.

In [2]:
from scipy.optimize import linprog

# Regressão L1: variáveis [b0, b1, u_1,...,u_m]
m = len(y)
c = np.r_[0., 0., np.ones(m)]
Aub = np.vstack([np.c_[A, -np.eye(m)], np.c_[-A, -np.eye(m)]])
bub = np.r_[y, -y]
sol = linprog(c, A_ub=Aub, b_ub=bub, bounds=[(None,None)]*2 + [(0,None)]*m)
print('L2:', X, 'L1:', sol.x[:2])

L2: [1.43333333 1.12      ] L1: [1.    1.025]


## 2ª VE prática

**Tarefa:** contaminar um conjunto de observações, aplicar teste global, Baarda, L2 e L1/Huber e comparar as soluções. **Entregáveis:** código, tabela de resíduos, gráfico comparativo e decisão técnica justificada.